# TRPG 正式流程 Notebook

这个 notebook 用来从产品流程角度检查当前运行时是否满足下面这条链路：

1. 玩家选择规则和剧本
2. 检查 `core_state.json / rule_state.json / scenario_state.json / agent_runtime_state.json`，缺失时决定是否自动生成
3. 调用 `Beginning Prompt` 开始角色创建 / 开场引导
4. 进入正式故事循环，并检查 Narrator 的上下文策略

说明：

- 某些格子会真实调用当前配置的文本模型
- `AUTO_PARSE_MISSING=True` 时，如果缺少 `rule_state.json` 或 `scenario_state.json`，会触发自动结构化生成
- 当前版本 **不会** 自动生成 `core_state.json` 和 `agent_runtime_state.json`


## Block 1: 初始化环境与辅助函数

这一格负责：

- 定位仓库根目录
- 重新加载 `trpg_runtime`
- 定义简单的状态表格展示
- 定义“状态文件检查”和“需求差距检查”的辅助函数


In [ ]:
from __future__ import annotations

import importlib
import inspect
import json
import sys
from datetime import datetime
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if not (candidate / 'Code').exists():
            continue
        if (candidate / 'Story').exists() or (candidate / 'story').exists() or (candidate / 'Prompt').exists():
            return candidate
    raise RuntimeError(f'无法从当前路径定位仓库根目录: {start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import trpg_runtime
import trpg_runtime.engine as trpg_runtime_engine
import trpg_runtime.models as trpg_runtime_models
import trpg_runtime.prompt_loader as trpg_runtime_prompt_loader

importlib.reload(trpg_runtime_models)
importlib.reload(trpg_runtime_prompt_loader)
importlib.reload(trpg_runtime_engine)
importlib.reload(trpg_runtime)

from trpg_runtime import MinimalTRPGEngine, PromptRepository, create_initial_state, parse_player_action

print('MinimalTRPGEngine 来源 =', inspect.getsourcefile(MinimalTRPGEngine))
print('PromptRepository 来源 =', inspect.getsourcefile(PromptRepository))


def _render_table(records: list[dict[str, object]]) -> HTML:
    if not records:
        return HTML('<p><em>empty</em></p>')

    headers = list(records[0].keys())
    rows = []
    for record in records:
        cells = ''.join(
            f"<td style='padding:6px 10px; border:1px solid var(--jp-border-color2, #444); vertical-align:top; white-space:pre-wrap'>{escape(str(record.get(header, '')))}</td>"
            for header in headers
        )
        rows.append(f'<tr>{cells}</tr>')

    header_html = ''.join(
        f"<th style='padding:6px 10px; border:1px solid var(--jp-border-color2, #444); text-align:left'>{escape(header)}</th>"
        for header in headers
    )
    table = (
        "<table style='border-collapse:collapse; width:100%; margin:8px 0'>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        "</table>"
    )
    return HTML(table)


def to_plain_data(value: object) -> object:
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='python')
    return value


def show_json(title: str, value: object) -> None:
    display(Markdown(f'### {title}'))
    payload = to_plain_data(value)
    display(HTML(f"<details><summary>查看 JSON</summary><pre>{escape(json.dumps(payload, ensure_ascii=False, indent=2))}</pre></details>"))


def get_story_root(root: Path) -> Path:
    for name in ('Story', 'story'):
        candidate = root / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError('未找到 Story/story 目录')


def state_file_records(root: Path, rule_code: str, story_code: str) -> list[dict[str, object]]:
    story_root = get_story_root(root)
    rule_dir = story_root / rule_code.upper() / 'Rule'
    story_dir = story_root / rule_code.upper() / 'Story' / story_code
    specs = [
        ('core_state.json', [rule_dir / 'core_state.json', story_dir / 'core_state.json'], 'No', 'shared setup'),
        ('rule_state.json', [rule_dir / 'rule_state.json'], 'Yes', 'rule runtime seed'),
        ('scenario_state.json', [story_dir / 'scenario_state.json'], 'Yes', 'story structure seed'),
        ('agent_runtime_state.json', [rule_dir / 'agent_runtime_state.json', story_dir / 'agent_runtime_state.json'], 'No', 'hidden agent bootstrap'),
    ]
    records = []
    for file_name, candidates, auto_parse, scope in specs:
        existing = [candidate for candidate in candidates if candidate.exists()]
        records.append(
            {
                'file': file_name,
                'supported_locations': '\n'.join(str(candidate) for candidate in candidates),
                'exists': bool(existing),
                'resolved_path': existing[0] if existing else '',
                'auto_parse_if_missing': auto_parse,
                'scope': scope,
            }
        )
    return records


def requirement_gap_report() -> list[dict[str, object]]:
    return [
        {
            'step': '1. 规则/剧本选择',
            'status': 'Implemented',
            'notes': '可通过 PromptRepository 列表 + 参数选择完成，当前 notebook 里手动变量切换。',
        },
        {
            'step': '2. 状态文件检查',
            'status': 'Partially implemented',
            'notes': 'rule/scenario 缺失时可自动生成；core/agent_runtime 缺失时当前只跳过，不自动生成。',
        },
        {
            'step': '3. Beginning Prompt 开角',
            'status': 'Partially implemented',
            'notes': '当前可以调用 generate_opening()，但“角色创建问答 -> 写回玩家卡”的专用流程尚未实现。',
        },
        {
            'step': '4. 故事循环',
            'status': 'Implemented with gaps',
            'notes': 'Dicer/NPC/Narrator/Director 主循环已可运行，Narrator 流式已支持，但没有单独的用户/AI对话滑动窗口。',
        },
        {
            'step': 'Narrator 滑动上下文窗口',
            'status': 'Not implemented',
            'notes': '当前 Narrator 只吃 state + 本轮结果 + director_state，不直接吃最近 5 组用户/AI 对话。',
        },
    ]


def save_agent_transcript(root: Path, engine, rule_code: str, story_code: str, turn_logs: list[dict[str, object]]) -> Path:
    output_dir = root / 'test' / 'play_logs'
    output_dir.mkdir(parents=True, exist_ok=True)
    session_id = getattr(engine.state.meta, 'session_id', 'session')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = output_dir / f'{rule_code}_{story_code}_{session_id}_{timestamp}.txt'

    lines = [
        f'Rule: {rule_code}',
        f'Story: {story_code}',
        f'Session: {session_id}',
        f'Turn Count: {len(turn_logs)}',
        '',
    ]

    for record in turn_logs:
        result = record['turn_result']
        director_output = record.get('director_output')
        lines.extend([
            '=' * 28,
            f"Turn {record['turn_index']}",
            '=' * 28,
            f"Player Input: {record['player_text']}",
            '',
            '[Dicer Output]',
            json.dumps(to_plain_data(result.dicer_result), ensure_ascii=False, indent=2),
            '',
            '[NPC Manager Output]',
            json.dumps(to_plain_data(result.npc_result), ensure_ascii=False, indent=2),
            '',
            '[Narrator Output]',
            result.narration,
            '',
            '[Director State Used This Turn]',
            json.dumps(to_plain_data(result.director_state_used), ensure_ascii=False, indent=2),
            '',
            '[Director Output For Next Turn]',
            json.dumps(to_plain_data(director_output), ensure_ascii=False, indent=2) if director_output is not None else 'None',
            '',
        ])

    output_path.write_text('\n'.join(lines), encoding='utf-8')
    return output_path


## Block 2: 配置流程参数

这里决定：

- 玩家要玩的规则与剧本
- 是否允许缺失的 `rule/scenario` 文件自动调用 LLM 生成
- 是否真正执行 `Beginning Prompt`
- 是否真正进入交互式故事循环

说明：这个 notebook 不再使用预设 `PLAYER_INPUT`，真正的玩家输入会在后面的循环里现场输入。


In [ ]:
RULE_CODE = 'ADV'
STORY_CODE = 'THE_DUST'
PLAYER_NAME = '调查员'

AUTO_PARSE_MISSING = True
RUN_BEGINNING_PROMPT = True
RUN_STORY_LOOP = True
BACKGROUND_DIRECTOR = True
MAX_TURNS = 20

VISIBLE_NPCS = []
INTERACTIVE_OBJECTS = ['遗迹入口', '坍塌石柱', '沙地脚印']
HAZARDS = ['风沙视野受限']
NPC_STATES = {}


## Block 3: 列出可用规则与剧本

这一步对应“玩家先选择要玩什么”。当前 notebook 用变量切换，不是 UI 下拉框。

In [ ]:
repo = PromptRepository()
print('可用规则 =', repo.list_rule_codes())
print(f'{RULE_CODE} 可用剧本 =', repo.list_story_codes(RULE_CODE))


## Block 4: 检查状态文件是否存在

这一格展示当前产品需求里的 4 类状态文件，并说明缺失时当前系统的行为。

注意：

- `rule_state.json` 缺失：当前支持自动生成
- `scenario_state.json` 缺失：当前支持自动生成
- `core_state.json` 缺失：当前不自动生成
- `agent_runtime_state.json` 缺失：当前不自动生成


In [ ]:
records = state_file_records(ROOT, RULE_CODE, STORY_CODE)
display(_render_table(records))


## Block 5: 需求差距检查

这一格把你提出的流程需求，和当前代码已经实现到什么程度，做一个快速对照。

In [ ]:
display(_render_table(requirement_gap_report()))


## Block 6: 加载 ScenarioBundle，并决定是否自动补全 rule/scenario

这一格会：

- 读取规则文本、剧本文本、4 个 agent prompt、Beginning prompt
- 根据 `AUTO_PARSE_MISSING` 决定缺失时是否自动生成 `rule/scenario`

注意：

- 如果 `AUTO_PARSE_MISSING=True` 且缺少 `rule_state.json / scenario_state.json`，这里会真实调用 LLM
- `core_state.json / agent_runtime_state.json` 即便缺失，也不会自动生成


In [ ]:
scenario_bundle = repo.load_scenario(RULE_CODE, STORY_CODE)
state_overrides = repo.load_state_overrides(
    RULE_CODE,
    STORY_CODE,
    scenario=scenario_bundle,
    auto_parse_missing=AUTO_PARSE_MISSING,
)

show_json('ScenarioBundle 概览', {
    'rule_code': scenario_bundle.rule_code,
    'story_code': scenario_bundle.story_code,
    'title': scenario_bundle.title,
    'opening_scene': scenario_bundle.opening_scene,
})
show_json('state_overrides', state_overrides)


## Block 7: 创建引擎并查看初始状态

这里不会强制要求通过 `from_prompt_files()` 创建引擎，而是显式把 `state_overrides` 注入进去，方便看清初始化流程。

In [ ]:
initial_state = create_initial_state(
    scenario=scenario_bundle,
    player_name=PLAYER_NAME,
    visible_npcs=VISIBLE_NPCS,
    interactive_objects=INTERACTIVE_OBJECTS,
    hazards=HAZARDS,
    npc_states=NPC_STATES,
    state_overrides=state_overrides,
)

engine = MinimalTRPGEngine(
    scenario=scenario_bundle,
    state=initial_state,
    prompt_repository=repo,
)

show_json('初始 state', engine.state)
print('玩家输入将在 Block 9 的交互式循环里实时输入。')


## Block 8: Beginning Prompt / 角色创建入口

当前代码里，这一步对应的是 `engine.generate_opening()`。

现状判断：

- 已实现：能用 `Beginning Prompt` 生成开场 / 开角引导文案
- 未实现：专门的“角色创建问答 -> 写回 player/rule/core”流程

如果 `RUN_BEGINNING_PROMPT=False`，这一格只说明当前缺口；改成 `True` 才会真实调用模型。


In [ ]:
if RUN_BEGINNING_PROMPT:
    opening = engine.generate_opening()
    display(Markdown('### Beginning Prompt 输出'))
    print(opening)
else:
    print('已跳过真实模型调用。把 RUN_BEGINNING_PROMPT 改成 True 后可生成开场/开角文案。')


## Block 9: 正式故事循环（流式 Narrator）

这一步对应真正的回合运行。

当前推荐链路：

- `Dicer` 与 `NPC Manager` 并行
- `Narrator` 走 `stream_turn()` 做流式输出
- `Director` 在回合末后台准备下一轮

交互命令：

- 输入普通文本：按玩家发言推进一轮
- 输入 `/state`：查看当前 state
- 输入 `/director`：查看当前已提交的 Director state
- 输入 `/quit`：结束循环，并把本次游玩的 agent 输出导出成 txt

如果 `RUN_STORY_LOOP=False`，这一格不会真实调用模型。


In [ ]:
if RUN_STORY_LOOP:
    turn_index = 1
    turn_logs = []
    transcript_path = None
    while turn_index <= MAX_TURNS:
        player_text = input(f'[{turn_index}/{MAX_TURNS}] 玩家输入（/state, /director, /quit）> ').strip()
        if not player_text:
            print('请输入内容，或者使用 /quit 结束。')
            continue
        if player_text.lower() in {'/quit', '/exit'}:
            if turn_logs:
                transcript_path = save_agent_transcript(ROOT, engine, RULE_CODE, STORY_CODE, turn_logs)
                print(f'故事循环已结束，记录已保存到: {transcript_path}')
            else:
                print('故事循环已结束，本次没有可保存的回合记录。')
            break
        if player_text.lower() == '/state':
            show_json('当前 engine.state', engine.state)
            continue
        if player_text.lower() == '/director':
            show_json('当前 Director state', engine.state.director)
            continue

        display(Markdown(f'### Turn {turn_index}'))
        print('玩家输入 =', player_text)
        print('\nNarrator 输出:')

        final_result = None
        for event in engine.stream_turn(player_text, background_director=BACKGROUND_DIRECTOR):
            if event.event == 'narration_chunk':
                print(event.delta, end='', flush=True)
            elif event.event == 'turn_result':
                final_result = event.result

        director_output_for_log = final_result.next_director_result if final_result is not None else None
        if BACKGROUND_DIRECTOR and director_output_for_log is None:
            director_output_for_log = engine.collect_director_update(wait=True)

        turn_logs.append({
            'turn_index': turn_index,
            'player_text': player_text,
            'turn_result': final_result,
            'director_output': director_output_for_log,
        })

        print('\n\n--- Turn Result ---')
        show_json('本回合 TurnResult', final_result)
        if director_output_for_log is not None:
            show_json('本回合 Director Output', director_output_for_log)
        turn_index += 1
    else:
        if turn_logs:
            transcript_path = save_agent_transcript(ROOT, engine, RULE_CODE, STORY_CODE, turn_logs)
            print(f'已到达 MAX_TURNS={MAX_TURNS}，循环结束。记录已保存到: {transcript_path}')
        else:
            print(f'已到达 MAX_TURNS={MAX_TURNS}，循环结束。')
else:
    print('已跳过真实故事循环。把 RUN_STORY_LOOP 改成 True 后可进入交互式循环。')


## Block 10: Narrator 是否需要滑动上下文窗口

当前实现：

- Narrator 会拿到 `state.core / state.rule / state.scenario`
- 会拿到本轮 `dicer_result / npc_result`
- 会拿到上一轮已提交的 `director_state`
- **不会** 直接拿最近 5 组用户/AI 原文对话

我对这个需求的判断：

- 从“叙事连贯性”看，加一个滑动窗口是有价值的
- 从“token 成本和信息污染”看，不建议直接塞满 10 条原文
- 更推荐：
  1. 保留最近 `2~3` 组原文对话窗口
  2. 更老的历史折叠进 `chapter_summary`
  3. 关键选择进 `player_choices`
  4. 客观事实进 `recent_events / event_records`

也就是说，**不建议一上来硬塞最近 5 组原文给 Narrator**，更稳的是“小窗口 + 状态摘要”。

这部分当前还没有实现成单独的 `conversation window` 字段。
